<a href="https://colab.research.google.com/github/momo4201/medical_imaging_and_informatics/blob/main/foundation_model_radiology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Foundation models and its applications**
A foundation model in radiology is a large neural network pretrained on massive medical image datasets (X-rays, CT, MRI) that learns general visual and anatomical features.

Instead of training from scratch, you fine-tune the model for a specific task such as:

*   Thoracic diseases detection (Pneumonia, Tuberculosis, Cardiomegaly, etc.)
*   Tumor segmentation
*   Outcome Prediction

Why foundation models matter in radiology?

*   Medical datasets are small & expensive
*   Pretraining captures anatomy and pathology patterns
*   Better performance with fewer labeled samples
*   Faster convergence and more stable training










**1️⃣ NIH ChestX-ray14 Models**

Task: Chest X-ray (14 diseases)

Modality: X-ray

Type: CNN-based foundation models

In [ ]:
from torchvision import models

model = models.densenet121(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 96.4MB/s]


2️⃣ TorchXRayVision

In [ ]:
!pip install torchxrayvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 14.4 MB/s eta 0:00:00


**✔ Pretrained on:**

*   ChestX-ray14
*   CheXpert
*   MIMIC-CXR
*   PadChest

In [ ]:
import torchxrayvision as xrv

model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.eval()

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]


XRV-DenseNet121-densenet121-res224-all

**5️⃣ BioViL (Microsoft)**

Type: Vision–Language Foundation Model

Modalities: X-ray + Radiology reports

In [ ]:
!pip install transformers

from transformers import AutoModel
model = AutoModel.from_pretrained("microsoft/biovil-base-pubmed")

**6️⃣ MedCLIP**

Type: CLIP-style foundation model

Strength: Zero-shot classification

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained("microsoft/medclip-resnet")

**7️⃣ MedSAM (Medical Adaptation of SAM)**

Task: Segmentation (X-ray, CT, MRI)

In [ ]:
!pip install opencv-python matplotlib torch torchvision tqdm
!git clone https://github.com/bowang-lab/MedSAM.git

Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Total 967 (delta 0), reused 0 (delta 0), pack-reused 967 (from 1)
Receiving objects: 100% (967/967), 62.91 MiB | 16.83 MiB/s, done.
Resolving deltas: 100% (475/475), done.


In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import sys

sys.path.append("MedSAM")

from segment_anything import sam_model_registry, SamPredictor

# **CT / MRI FOUNDATION MODEL – MONAI**

In [ ]:
!pip install monai nibabel einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.7 MB/s eta 0:00:00


In [ ]:
import torch
import nibabel as nib
import numpy as np

from monai.networks.nets import SwinUNETR
from monai.transforms import (
    ScaleIntensityRange,
    Resize
)

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
# Load NIfTI CT
nii = nib.load("/content/001_0000.nii.gz")
ct = nii.get_fdata()

print("Original shape:", ct.shape)

Original shape: (302, 302, 276)


In [ ]:
# Normalize HU values
normalize = ScaleIntensityRange(
    a_min=-1000, a_max=400,
    b_min=0.0, b_max=1.0,
    clip=True
)

ct = np.expand_dims(ct, axis=0)
print(ct.shape)

ct = normalize(ct)
print(ct.shape)

# Resize to SwinUNETR input size
resize = Resize(spatial_size=(96, 96, 96))
ct = resize(ct)

print(ct.shape)

# Convert to tensor
ct = torch.tensor(ct).unsqueeze(0).unsqueeze(0).float()
ct = ct.cuda()

(1, 302, 302, 276)
torch.Size([1, 302, 302, 276])
torch.Size([1, 96, 96, 96])


/tmp/ipython-input-4257341367.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ct = torch.tensor(ct).unsqueeze(0).unsqueeze(0).float()


In [ ]:
!wget https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt

--2025-12-27 06:04:45--  https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/366729051/c7bc9f02-a8fb-4527-b311-e308fce79182?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-27T06%3A50%3A24Z&rscd=attachment%3B+filename%3Dmodel_swinvit.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-27T05%3A49%3A34Z&ske=2025-12-27T06%3A50%3A24Z&sks=b&skv=2018-11-09&sig=nPFIpd6PmoodWWAwi0lgluDH6TcQ%2BzLcpXUxTN2iXB0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NjgxOTA4NSwibmJmIjoxNzY2ODE1NDg1LCJwYXRoIjoicmVsZWFzZWF

In [ ]:
model = SwinUNETR(
    in_channels=1,
    out_channels=14,
    feature_size=48,
    use_checkpoint=True,
).cuda()

# Load pretrained weights
weight = torch.load("/content/model_swinvit.pt", weights_only=True)
model.load_from(weights=weight)
print("Using pretrained self-supervied Swin UNETR backbone weights !")

model.eval()

Using pretrained self-supervied Swin UNETR backbone weights !


SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(1, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=48, out_features=192, bias=True)
              (linear2): Linear(in_feature

In [ ]:
print(ct.shape)
# ct = torch.randn(1, 1, 96, 96, 96).cuda()

with torch.no_grad():
    hidden_states = model.swinViT(ct)

# Use last-stage features
features = hidden_states[-1]

print("Feature map shape:", features.shape)

torch.Size([1, 1, 96, 96, 96])
Feature map shape: torch.Size([1, 768, 3, 3, 3])
